In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score , roc_curve , classification_report

In [2]:
# --------GENERATE SYNTHETIC CREDIT BUREAU DATA ---------
print("Generating credit applicant profiles .....")
np.random.seed(42)
n_applicants = 4000
data = {
    'applicant_id':range(1, n_applicants + 1),
    'income':np.random.normal(loc=65000, scale=20000, size=n_applicants),
    'debt_to_income_ratio':np.random.uniform(0.1, 0.7, size=n_applicants),
    'missed_payment_last_2yr':np.random.poisson(lam=0.5, size=n_applicants),
    'credit_utilization_rate':np.random.uniform(0.05, 0.95, size=n_applicants),
    'defaulted':np.random.choice([0,1], size=n_applicants, p=[0.91,0.09])
}
df = pd.DataFrame(data)
# Enforce realistic credit rules 
risk_score = (df['debt_to_income_ratio']*3)+df['missed_payment_last_2yr']+(df['credit_utilization_rate']*2)
df.loc[risk_score>3.5, 'defaulted'] = np.random.choice([0,1], size=len(df[risk_score>3.5]), p=[0.2,0.8])
# Clean up negative incomes if any
df['income'] = df['income'].clip(lower=15000)

Generating credit applicant profiles .....


In [3]:
# --------- PREPROCESS AND SCALE FEATURES -----------
X = df[['income','debt_to_income_ratio','missed_payment_last_2yr','credit_utilization_rate']]
y = df['defaulted']
# Split data into 75% Training and 25% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# ------- TRAIN CREDIT RISK MODEL ---------
print("Training Risk Engine (Logistic Regression) ....")
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

Training Risk Engine (Logistic Regression) ....


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [5]:
# ------ GENERATE A COMPLETE 'CREDIT' SCORE --------
# Instead of a binary 0 or 1 , banks want a risk probability score 
# Let's map the probability of NOT defaulting to a traditional 300-850 Credit Score Scale
probabilities = model.predict_proba(X_test_scaled)[:,0] # Probability of class 0 (Good Borrower)
# Scale to 300-850 FICO Style score 
credit_scores = 300 + (probabilities*550)
X_test_with_scores = X_test.copy()
X_test_with_scores['predicted_credit_score'] = credit_scores.astype(int)
X_test_with_scores['actual_default'] = y_test
print("\n--- Sample Generated Credit Scores ---")
print(X_test_with_scores[['income','credit_utilization_rate','predicted_credit_score','actual_default']].head())


--- Sample Generated Credit Scores ---
            income  credit_utilization_rate  predicted_credit_score  \
2231  80812.511983                 0.498762                     545   
1328  40806.105143                 0.750807                     725   
43    58977.926088                 0.822212                     514   
1251  81670.579232                 0.360206                     826   
3049  88944.938473                 0.380206                     674   

      actual_default  
2231               0  
1328               0  
43                 0  
1251               0  
3049               0  


In [8]:
# ------- EVALUATE MODEL USING ROC-AUC -------
# ROC-AUC is the gold-standard metric for credit scoring models
y_pred_proba = model.predict_proba(X_test_scaled)[:,1] # Prob of default
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"\nModel ROC-AUC : {auc_score:.4f}")


Model ROC-AUC : 0.7984


In [ ]:
# Plot the ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score:.2f})')
plt.plot([0,1], [0,1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Credit Risk Model ROC Curve')
plt.legend(